# 00 — Collecte des données : scraping des nominations Oscar (Wikipédia)

> **Point d'entrée du pipeline.** Ce notebook constitue la *source de vérité* des nominations :
> il scrape les pages Wikipédia « *Nth Academy Awards* » (1929–2026) et en extrait, pour chaque
> cérémonie, la liste des catégories avec gagnant·e et nommé·e·s. Le résultat alimente
> [`01_Merge_datasets.ipynb`](01_Merge_datasets.ipynb), qui le croise avec IMDb et TMDb.

## 1. Scraper Wikipédia — parsing des pages de cérémonie

La classe `Award` et les helpers de parsing nettoient le HTML (espaces parasites, parenthèses,
notes de bas de page) et reconstruisent, page par page, la structure `cérémonie → catégories → (gagnant, nommés)`. La fonction `scrape_all_wiki()` itère sur toutes
les cérémonies.

In [10]:
from bs4 import BeautifulSoup
import requests
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
import logging
from tqdm import tqdm


logging.basicConfig(level = logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class Award:
    category: str
    winner: str
    nominees: List[str]

def clean_spaces(text: str) -> str:
    """
    Removes whitespace between quotation marks and parentheses when parsing.

    Ie. Parsing results in ( Brazil ), cleaned to (Brazil)
    """

    text = re.sub(r'"\s*([^"]*?)\s*"', r'"\1"', text)
    text = re.sub(r'\(\s*([^)]*?)\s*\)', r'(\1)', text)
    return text

def generate_oscar_urls(start_year: int = 1929, end_year: int = 2026) -> List[Dict]:
    """
    Generate URLs for every Oscar ceremony from start_year to end_year.
    """

    urls = []

    for year in range(start_year, end_year + 1):
        ceremony_num = year - 1928
        if 10 <= ceremony_num % 100 <= 13:
            suffix = "th"
        else:
            last_digit = ceremony_num % 10
            suffix = {
                    1: "st",
                    2: "nd",
                    3: "rd"
            }.get(last_digit, "th")

        url = f"https://en.wikipedia.org/wiki/{ceremony_num}{suffix}_Academy_Awards"

        urls.append({
                "ceremony_number": ceremony_num,
                "year": year,
                "url": url,
        })

    return urls

class WikipediaScraper:
    """
    Scrapes data from Oscars Wikipedia pages.
    """

    # ! Awards table format is different for 18th (1946) - 49th (1977) awards. It uses separate cells for the awards/categories vs. it being in the same cell as the winners/nominees.
    # ? This issue doesn't appear to be a thing anymore after 10 months coming back to this (a/o Feb 11, 2026)

    def __init__(self, url: str):
        self.url = url
        self.session = requests.Session()
        self.session.headers.update({
                "User-Agent": (
                        "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                        "AppleWebKit/537.36 (KHTML, like Gecko)"
                        "Chrome/134.0.0.0 Safari/537.36"
                        )
        })
        self.soup = self.get_soup(self.url)

    def get_soup(self, url: str) -> BeautifulSoup:
        """
        Fetches and parses the HTML from the Wikipedia url.
        """

        try:
            response = self.session.get(url, allow_redirects = True)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        except requests.RequestException as e:
            logger.error(f"Failed to fetch {self.url}: {e}")
            raise

    def scrape_ceremony(self) -> Dict:
        """
        Scrape data from a single Oscar ceremony Wikipedia page.
        """

        logger.info(f"Scraping: {self.url}")

        ceremony_data = {
                "url": self.url,
                "date": self.get_date(),
                "awards": []
        }

        try:
            award_table_rows = self.get_award_table_rows()

            for row in award_table_rows[:]:
                cells = row.find_all("td")
                for cell in cells:
                    award_data = Award(
                            category = self.get_award(cell),
                            winner = self.get_winner(cell),
                            nominees = self.get_nominees(cell)
                    )
                    ceremony_data["awards"].append({
                        "category": award_data.category,
                        "winner": award_data.winner, 
                        "nominees": award_data.nominees
                    })

        except Exception as e:
            logger.error(f"Failed to parse ceremony {self.url}: {e}")

        return ceremony_data

    def get_award_table_rows(self) -> List[BeautifulSoup]:
        """
        Extracts the Awards table and its corresponding rows for subsequent parsing. This could be titled differently across different pages.
        """
        
        
        possible_titles = ["Awards", "Winners and nominees"]

        for header_tag in ["h3", "h2"]:
            for title in possible_titles:
                header = self.soup.find(header_tag, string = lambda s: s and s.lower().strip() == title.lower())
                if header:
                    table = header.find_next("table", class_ = "wikitable")
                    if table:
                        return table.find_all("tr") # tr for rows.

        raise Exception("Awards section not found with expected headers.")

    def get_date(self) -> Optional[str]:
        """
        Extracts the year of the Academy Awards ceremony the Wikipedia infobox.
        """

        infobox = self.soup.find("table", class_ = "infobox vevent")
        if not infobox:
            return None

        for row in infobox.find_all("tr"):
            header = row.find("th") # th for header.
            if header and "Date" in header.text:
                date_cell = row.find("td")
                if date_cell:
                    return date_cell.get_text(strip = True)

        return None

    def get_award(self, table_cell) -> str:
        """
        Extracts the award name from a table cell element. Returns "Unknown Award" if no award name is found.
        """

        award_div = table_cell.find("div")
        award_name = award_div.get_text(strip = True) if award_div else "Unknown Award"
        return award_name

    def split_nominee_text(self, nominee_text: str) -> Tuple[str, Optional[str]]:
        """
        Takes the nominee line item information and returns the primary winner and secondary information.

        Ie. In the 97th Oscars (2025):
        For Best Picture, Anora is the primary and Sean Baker (+ producers) is in the secondary.
        For Best Director, Sean Baker is the primary and Anora is the secondary.
        """

        if " – " in nominee_text:
            primary, secondary = nominee_text.split(" – ", 1)
        else:
            primary, secondary = nominee_text, None
        return [
                clean_spaces(primary.strip()),
                clean_spaces(secondary.strip()) if secondary else None
        ]

    def get_winner(self, table_cell) -> str:
        """
        Extracts the winner from the Wikipedia Awards table cells - usually bolded inside <ul><li>.
        """

        try:
            ul = table_cell.find("ul")
            main_li = ul.find("li")
            winner_text = main_li.find("b").get_text(separator = " ", strip = True)
            return self.split_nominee_text(winner_text)[0]
        except (AttributeError, TypeError):
            return "Unknown Winner"

    def get_nominees(self, table_cell) -> List[str]:
        """
        Extracts the nominees from the Wikipedia Awards table cell.
        Assumes:
            - Winner is in the first <li>
            - Nominees (if any) are in a nested <ul> inside that first <li>
        """
        nominees = []

        try:
                # find all <ul> elements in the cell
                ul_elements = table_cell.find_all("ul")

                if not ul_elements or len(ul_elements) < 1:
                        logger.warning("No <ul> found in table_cell: %s", table_cell.get_text(strip = True))
                        return []

                # first <ul> contains the winner in <li>, and possibly a nested <ul> with nominees
                first_ul = ul_elements[0]
                first_li = first_ul.find("li")

                if not first_li:
                        logger.warning("No <li> found in first <ul>: %s", first_ul)
                        return []

                # look for a nested <ul> inside the winner <li> (this contains nominees)
                sub_ul = first_li.find("ul")

                if not sub_ul:
                        logger.info("No nominees <ul> found in winner <li>; only winner present.")
                        return []

                # Each <li> in sub_ul is a nominee
                nominee_items = sub_ul.find_all("li")
                for nominee in nominee_items:
                        nominee_text = nominee.get_text(separator=" ", strip=True)
                        parsed_name = self.split_nominee_text(nominee_text)[0]
                        nominees.append(parsed_name)

        except Exception as e:
                logger.warning("Unexpected error while extracting nominees: %s", str(e))

        return nominees

def scrape_all_wiki(start_year: int = 1929, end_year: int = 2026):
    urls = generate_oscar_urls(start_year, end_year)
    all_data = []

    for entry in tqdm(urls, desc = "Scraping Oscars Wiki pages"):
        try:
            scraper = WikipediaScraper(entry["url"])
            data = scraper.scrape_ceremony()
            data["ceremony_number"] = entry["ceremony_number"]
            data["year"] = entry["year"]
            all_data.append(data)
        except Exception as e:
            logger.error(f"Failed to scrape {entry['url']}: {e}")

    return all_data

def main(start_year: int = 1929, end_year: int = 2026):
    all_data = scrape_all_wiki(start_year, end_year)

    for ceremony in all_data:
        for award in ceremony["awards"]:
            if isinstance(award, dict):
                category = award.get("category", "Unknown Award")
                winner = award.get("winner", "Unknown Winner")
                nominees = award.get("nominees", [])
            else:
                category = getattr(award, "category", "Unknown Award")
                winner = getattr(award, "winner", "Unknown Winner")
                nominees = getattr(award, "nominees", [])

            print(
                ceremony["year"], 
                category,
                winner, 
                nominees
            )

if __name__ == "__main__":
    main()

Scraping Oscars Wiki pages:   0%|          | 0/98 [00:00<?, ?it/s]INFO:__main__:Scraping: https://en.wikipedia.org/wiki/1st_Academy_Awards
INFO:__main__:No nominees <ul> found in winner <li>; only winner present.
Scraping Oscars Wiki pages:   4%|▍         | 4/98 [00:00<00:14,  6.31it/s]INFO:__main__:Scraping: https://en.wikipedia.org/wiki/5th_Academy_Awards
INFO:__main__:No nominees <ul> found in winner <li>; only winner present.
Scraping Oscars Wiki pages:   5%|▌         | 5/98 [00:00<00:15,  6.15it/s]INFO:__main__:Scraping: https://en.wikipedia.org/wiki/6th_Academy_Awards
INFO:__main__:No nominees <ul> found in winner <li>; only winner present.
Scraping Oscars Wiki pages:  14%|█▍        | 14/98 [00:02<00:13,  6.41it/s]INFO:__main__:Scraping: https://en.wikipedia.org/wiki/15th_Academy_Awards
INFO:__main__:No nominees <ul> found in winner <li>; only winner present.
Scraping Oscars Wiki pages:  21%|██▏       | 21/98 [00:03<00:12,  6.17it/s]INFO:__main__:Scraping: https://en.wikipedia.or

1929 Outstanding Picture Wings ['The Racket', '7th\xa0Heaven']
1929 Best Unique and Artistic Picture Sunrise ['Chang', 'The Crowd']
1929 Best Directing (Dramatic Picture) Frank Borzage ['Herbert Brenon', 'King Vidor']
1929 Best Directing (Comedy Picture) Lewis Milestone ['Charles Chaplin', 'Ted Wilde']
1929 Best Actor Emil Jannings ['Richard Barthelmess', 'Charles Chaplin']
1929 Best Actress Janet Gaynor ['Louise Dresser', 'Gloria Swanson']
1929 Best Writing (Adaptation) 7th Heaven ['Glorious Betsy', 'The Jazz Singer']
1929 Best Writing (Original Story) Underworld ['The Circus', 'The Last Command']
1929 Best Writing (Title Writing) Joseph Farnham ['Gerald Duffy [ B ]', 'George Marion Jr.']
1929 Best Art Direction The Dove []
1929 Best Cinematography Sunrise ['The Devil Dancer', 'The Magic Flame', 'Sadie Thompson']
1929 Best Engineering Effects Wings ['No specific film', 'No specific film']
1930 Outstanding Picture The Broadway Melody ['Alibi', 'The Hollywood Revue of 1929', 'In Old Ari

## 2. Mise en forme tabulaire & export

On aplatit la structure imbriquée en un DataFrame « une ligne = une catégorie × cérémonie »
(les nommés sont concaténés par `|`), puis on exporte en **CSV** *et* **JSON** dans
`Data/Raw/Scraping/`. Ces fichiers (`all_data_oscars.*`) sont la matière première du merge.

In [15]:
from pathlib import Path
import pandas as pd

# Transforme all_data en un seul dataset tabulaire et l'enregistre

# Si all_data n'existe pas encore, on le scrape
if "all_data" not in globals():
    all_data = scrape_all_wiki()

rows = []
for ceremony in all_data:
    for award in ceremony.get("awards", []):
        rows.append({
            "ceremony_number": ceremony.get("ceremony_number"),
            "year": ceremony.get("year"),
            "date": ceremony.get("date"),
            "url": ceremony.get("url"),
            "category": award.get("category"),
            "winner": award.get("winner"),
            "nominees": " | ".join(award.get("nominees", []))
        })

df_all_data = pd.DataFrame(rows)

output_dir = Path("../Data/Raw/Scraping")
output_dir.mkdir(parents=True, exist_ok=True)

df_all_data.to_csv(output_dir / "all_data_oscars.csv", index=False, encoding="utf-8")
df_all_data.to_json(output_dir / "all_data_oscars.json", orient="records", force_ascii=False, indent=2)

print(f"Fichiers enregistrés dans : {output_dir.resolve()}")
print(f"Lignes: {len(df_all_data)}")

Fichiers enregistrés dans : /Users/jo/Desktop/ML For Business/applied_ml_for_business/Data/Raw/Scraping
Lignes: 2194


## 3. Conclusion

- **Sortie** : `Data/Raw/Scraping/all_data_oscars.csv` / `.json` — 2 194 lignes (1929–2026),
  colonnes `ceremony_number, year, date, url, category, winner, nominees`.
- **Limite assumée** : le format Wikipédia varie selon les époques ; le parsing est tolérant
  (nettoyage d'espaces / parenthèses) mais la normalisation fine des 119 catégories historiques
  (regroupement des variantes, passage en format long) est faite en aval dans
  [`01_Merge_datasets.ipynb`](01_Merge_datasets.ipynb).
- **Aval du pipeline** : ces nominations sont croisées avec IMDb (films & personnes) et TMDb dans [`01_Merge_datasets.ipynb`](01_Merge_datasets.ipynb).
